# Apply Sentiment Labels to All Forum Posts

## Imports

In [1]:
import gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive

## Mount Google Drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


## Config

In [3]:
MODEL_PATH        = '/content/drive/MyDrive/ColabThesisData/models/bert-large-finnish-cased-v1_final_2026-03-22_12-13-20/'
ALL_POSTS_PATH    = '/content/drive/MyDrive/ColabThesisData/cleaned_forum_posts.parquet'
LLM_LABELS_PATH   = '/content/drive/MyDrive/ColabThesisData/llm_labeled.parquet'
HUMAN_LABELS_PATH = '/content/drive/MyDrive/ColabThesisData/labeled.parquet'
OUTPUT_PATH       = '/content/drive/MyDrive/ColabThesisData/all_labeled_finnishbert.parquet'

LABEL_NAMES    = ['negative', 'neutral', 'positive']
MAX_SEQ_LENGTH = 512
BATCH_SIZE     = 256
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

Device: cuda


## Load Data

In [4]:
all_posts    = pd.read_parquet(ALL_POSTS_PATH)
llm_labels   = pd.read_parquet(LLM_LABELS_PATH)
human_labels = pd.read_parquet(HUMAN_LABELS_PATH)

print(f'All forum posts : {len(all_posts):,}')
print(f'LLM labels      : {len(llm_labels):,}')
print(f'Human labels    : {len(human_labels):,}')

All forum posts : 532,424
LLM labels      : 10,000
Human labels    : 608


## Identify Already-labeled Posts

In [5]:
all_posts['id'] = all_posts['id'].astype(str)
llm_labels['id']   = llm_labels['id'].astype(str)
human_labels['id'] = human_labels['id'].astype(str)

already_labeled = set(llm_labels['id']) | set(human_labels['id'])
to_infer = all_posts[~all_posts['id'].isin(already_labeled)].reset_index(drop=True)

print(f'Already labeled : {len(already_labeled):,}')
print(f'Needs inference : {len(to_infer):,}')

Already labeled : 10,608
Needs inference : 521,816


## Load Model

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.eval()
print('Model loaded.')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Model loaded.


## Run Inference

In [7]:
class TextDataset(TorchDataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return self.encodings['input_ids'].shape[0]
    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}


# Format text the same way as during training
texts = ('yritys: ' + to_infer['company_name'] + ' viesti: ' + to_infer['message']).tolist()

encodings = tokenizer(
    texts,
    truncation=True,
    max_length=MAX_SEQ_LENGTH,
    padding=True,
    return_tensors='pt',
)
loader = DataLoader(TextDataset(encodings), batch_size=BATCH_SIZE)

all_preds = []
total = len(loader)

with torch.no_grad():
    for i, batch in enumerate(loader, 1):
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        preds  = torch.argmax(model(**batch).logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        if i % 50 == 0 or i == total:
            print(f'  {i}/{total} batches done')

print(f'Inference complete: {len(all_preds):,} predictions')

  50/2039 batches done
  100/2039 batches done
  150/2039 batches done
  200/2039 batches done
  250/2039 batches done
  300/2039 batches done
  350/2039 batches done
  400/2039 batches done
  450/2039 batches done
  500/2039 batches done
  550/2039 batches done
  600/2039 batches done
  650/2039 batches done
  700/2039 batches done
  750/2039 batches done
  800/2039 batches done
  850/2039 batches done
  900/2039 batches done
  950/2039 batches done
  1000/2039 batches done
  1050/2039 batches done
  1100/2039 batches done
  1150/2039 batches done
  1200/2039 batches done
  1250/2039 batches done
  1300/2039 batches done
  1350/2039 batches done
  1400/2039 batches done
  1450/2039 batches done
  1500/2039 batches done
  1550/2039 batches done
  1600/2039 batches done
  1650/2039 batches done
  1700/2039 batches done
  1750/2039 batches done
  1800/2039 batches done
  1850/2039 batches done
  1900/2039 batches done
  1950/2039 batches done
  2000/2039 batches done
  2039/2039 batches 

## Combine Inferred + Existing Labels

In [8]:
inferred_df = to_infer.copy()
inferred_df['sentiment']    = all_preds
inferred_df['label_source'] = 'model'

llm_out             = llm_labels.copy()
llm_out['label_source'] = 'llm'

human_out               = human_labels.copy()
human_out['label_source'] = 'human'

keep_cols = ['id', 'message', 'company_name', 'sentiment', 'label_source']

combined = pd.concat(
    [inferred_df[keep_cols], llm_out[keep_cols], human_out[keep_cols]],
    ignore_index=True,
)

print(f'Total rows : {len(combined):,}')
print(combined['label_source'].value_counts())
print()
print(combined['sentiment'].value_counts().sort_index().rename({0: 'negative', 1: 'neutral', 2: 'positive'}))

Total rows : 532,424
label_source
model    521816
llm       10000
human       608
Name: count, dtype: int64

sentiment
negative    144671
neutral     185584
positive    202169
Name: count, dtype: int64


## Save Output

In [9]:
combined.to_parquet(OUTPUT_PATH, index=False)
print(f'Saved {len(combined):,} rows to {OUTPUT_PATH}')

del model
gc.collect()
torch.cuda.empty_cache()

Saved 532,424 rows to /content/drive/MyDrive/ColabThesisData/all_labeled_finnishbert.parquet
